# Okun Bible PDF — Page Image Extractor

This notebook processes scanned PDF files where:
- Each PDF page contains a **landscape photo** rotated to portrait
- Each photo contains **2 bible pages** (left and right)

**Output:** Individual high-resolution PNG images of each page, ready for text extraction.

---
### Workflow
1. Install dependencies
2. Mount Google Drive
3. Configure your PDF file paths
4. Run extraction
5. Preview results
6. Verify output file list

## Step 1 — Install Dependencies

In [ ]:
!pip install pymupdf pillow -q
print("Dependencies installed.")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## Step 3 — Configuration

> **Edit this cell before running anything else.**
> Update `PDF_FILES` with the actual paths to your 3 PDFs in Google Drive.
> Update `OUTPUT_DIR` if you want output saved elsewhere.

In [ ]:
# ── EDIT THESE PATHS ──────────────────────────────────────────────────────────

PDF_FILES = [
    '/content/drive/MyDrive/Colab Notebooks/Okun digitization/okun_bible_part1.pdf',  # <-- update
    '/content/drive/MyDrive/Colab Notebooks/Okun digitization/okun_bible_part2.pdf',  # <-- update
    '/content/drive/MyDrive/Colab Notebooks/Okun digitization/okun_bible_part3.pdf',  # <-- update
]

OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/okun_pages_images'

# DPI: 300 = sharp text, good for OCR. Lower to 200 if storage is tight.
DPI = 300

# ── DO NOT EDIT BELOW ─────────────────────────────────────────────────────────
import os
import fitz
import io
from PIL import Image

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Validate PDF files exist
print("Checking PDF file paths...")
all_ok = True
for path in PDF_FILES:
    exists = os.path.exists(path)
    status = "FOUND" if exists else "NOT FOUND"
    print(f"  [{status}] {path}")
    if not exists:
        all_ok = False

if all_ok:
    print(f"\nAll PDFs found. Output will be saved to:\n  {OUTPUT_DIR}")
else:
    print("\nFix the paths above before continuing.")

## Step 4 — Run Extraction

For each PDF page the script will:
1. Render at high DPI
2. Detect and correct portrait-rotated landscape images
3. Split down the middle → Page A (left) and Page B (right)
4. Save both as PNG files named: `pdf01_scan001_pageA.png`, `pdf01_scan001_pageB.png`, etc.

In [ ]:
import fitz
import io
import os
from PIL import Image

def process_pdf(pdf_path, pdf_index, dpi=300):
    """Extract, rotate, and split all scanned pages from one PDF."""
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    print(f"\n[PDF {pdf_index+1}] {os.path.basename(pdf_path)} — {total_pages} scanned pages")
    print(f"  Expected output: {total_pages * 2} individual page images")
    print("  Processing", end="", flush=True)

    saved_count = 0
    for page_num in range(total_pages):
        page = doc[page_num]

        # Render page at target DPI
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        img = Image.open(io.BytesIO(pix.tobytes("png")))

        w, h = img.size

        # If portrait (h > w), the landscape scan was rotated — fix it
        if h > w:
            # Rotate 90 degrees counter-clockwise to restore landscape
            img = img.rotate(90, expand=True)
            w, h = img.size

        # Split landscape image into left page and right page
        mid = w // 2
        left_page  = img.crop((0, 0, mid, h))
        right_page = img.crop((mid, 0, w, h))

        # Build output filenames
        base = f"pdf{pdf_index+1:02d}_scan{page_num+1:03d}"
        left_path  = os.path.join(OUTPUT_DIR, f"{base}_pageA.png")
        right_path = os.path.join(OUTPUT_DIR, f"{base}_pageB.png")

        left_page.save(left_path,  dpi=(dpi, dpi))
        right_page.save(right_path, dpi=(dpi, dpi))
        saved_count += 2

        print(".", end="", flush=True)

    doc.close()
    print(f" done ({saved_count} images saved)")
    return saved_count


# ── Run across all PDFs ───────────────────────────────────────────────────────
print("Starting extraction...")
total_saved = 0

for i, pdf_path in enumerate(PDF_FILES):
    if not os.path.exists(pdf_path):
        print(f"\nSkipping (not found): {pdf_path}")
        continue
    total_saved += process_pdf(pdf_path, i, dpi=DPI)

print(f"\nExtraction complete. Total images saved: {total_saved}")
print(f"Output folder: {OUTPUT_DIR}")

## Step 5 — Preview Sample Output

Displays the first 4 extracted page images so you can verify quality and orientation before downloading.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# Get all output PNGs sorted
all_images = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.png')))

if not all_images:
    print("No output images found. Run Step 4 first.")
else:
    preview_images = all_images[:4]  # Show first 4
    fig, axes = plt.subplots(1, len(preview_images), figsize=(20, 12))

    if len(preview_images) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, preview_images):
        img = mpimg.imread(img_path)
        ax.imshow(img, cmap='gray')
        ax.set_title(os.path.basename(img_path), fontsize=8)
        ax.axis('off')

    plt.suptitle('Sample Output — First 4 Extracted Pages', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f"Previewing {len(preview_images)} of {len(all_images)} total images.")

## Step 6 — Verify Output File List

In [ ]:
import glob

all_images = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.png')))

print(f"Total images in output folder: {len(all_images)}")
print(f"Output directory: {OUTPUT_DIR}")
print("\nFile listing:")
for img_path in all_images:
    size_kb = os.path.getsize(img_path) // 1024
    print(f"  {os.path.basename(img_path):40s}  {size_kb:>6} KB")

---
## Notes

- Output files are saved directly to your **Google Drive** and persist after the session ends.
- Naming convention: `pdf01_scan001_pageA.png` = PDF 1, scan 1, left page.
- If rotation looks wrong (some pages upside-down), the scan may have been rotated 90° clockwise instead. Change `rotate(90` to `rotate(-90` in Step 4.
- To re-run a single PDF without reprocessing others, change `PDF_FILES` in Step 3 to contain only that one file and re-run Steps 4–6.